# Juan PatchCore Lite Reproduction

In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.common.config import load_config
from src.common.data import SpacepressoDataModule
from src.common.paths import resolve_path
from src.common.q8rle import float_matrix_to_q8rle, q8rle_to_float_matrix
from src.common.seed import set_seed
from src.common.submission import _prepare_prediction_map
from src.common.training import ExperimentRunner
from src.common.visualization import show_predictions
from src.methods import get_method_class

In [ ]:
RUN_RETRAIN = False
RUN_PREDICT = True
WRITE_SUBMISSION = True
SEED = 42
set_seed(SEED)

config = load_config(ROOT / "configs/patchcore_lite/juan_full_ensemble_reproduction.yaml")
config["seed"] = SEED

data_root_candidates = [
    ROOT / "data" / "spacepresso" / "adl-2025-2026-anomaly-detection",
    ROOT / "data" / "spacepresso",
    Path("/content/data/spacepresso"),
]
resolved_data_root = next((path for path in data_root_candidates if path.exists()), None)
if resolved_data_root is not None:
    config["data"]["root"] = str(resolved_data_root)
config["data"]["load_images"] = False

print({"data_root": config["data"]["root"], "exists": Path(config["data"]["root"]).exists()})
print({
    "selected_folds": config["reproduction"]["selected_folds"],
    "metadata": config["reproduction"]["ensemble_metadata_path"],
    "known_submission_zip": config["reproduction"]["known_submission_zip"],
})

In [ ]:
dm = SpacepressoDataModule(**config["data"])
train_good = dm.load_train_good()
train_anomalies = dm.load_train_anomalies()
test = dm.load_test()
print({"train_good": len(train_good), "train_anomalies": len(train_anomalies), "test": len(test)})

In [ ]:
Method = get_method_class(config["method"]["name"])
runner = ExperimentRunner(Method(config), config)
output_dir = resolve_path(config["experiment"]["output_dir"], ROOT)
metadata_path = output_dir / "ensemble_metadata.json"

if RUN_RETRAIN or not metadata_path.exists():
    if not (train_good and train_anomalies):
        raise RuntimeError("Need both clean and anomalous training samples to retrain the ensemble.")
    print({"training_patchcore_ensemble": True, "output_dir": str(output_dir)})
    runner.fit(train_good, train_anomalies)
else:
    print({"training_patchcore_ensemble": False, "using_existing_metadata": str(metadata_path)})
    runner.method.load(output_dir)

fold_metrics = pd.DataFrame([item["record"] for item in runner.method.fold_records])
fold_metrics = fold_metrics.sort_values("score", ascending=False).reset_index(drop=True)
display(fold_metrics[["fold", "selected", "metric", "score", "pixel_ap", "image_ap", "pixel_auroc", "n_val_samples"]])
print({"selected_folds": [record["fold"] for record in runner.method.selected_records], "selected_count": len(runner.method.selected_records)})

In [ ]:
if RUN_PREDICT:
    predictions = runner.predict(test)
    preview_samples = sorted(
        [sample for sample in test if sample.image_id in predictions],
        key=lambda sample: float(np.max(predictions[sample.image_id])),
        reverse=True,
    )[:3]
    show_predictions(preview_samples, predictions, n=len(preview_samples), autoscale=True, show_stats=True)
    print({"predictions": len(predictions)})

In [ ]:
if RUN_PREDICT and WRITE_SUBMISSION and predictions:
    expected_shape = (config["data"]["image_size"], config["data"]["image_size"])
    sorted_ids = sorted(predictions)
    assert len(sorted_ids) == len(test), f"Expected {len(test)} predictions, got {len(sorted_ids)}"

    sample_indices = set(random.sample(range(len(sorted_ids)), min(5, len(sorted_ids))))
    sampled_labels = {}
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_path = resolve_path(config["submission"]["output_path"], ROOT)
    zip_path = base_path.parent / f"{base_path.stem}_{timestamp}.zip"
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        with zf.open(base_path.stem + ".csv", "w") as f:
            f.write(b"ID,Label\n")
            for i, image_id in enumerate(sorted_ids):
                label = float_matrix_to_q8rle(_prepare_prediction_map(predictions[image_id]))
                f.write(f"{image_id},{label}\n".encode("utf-8"))
                if i in sample_indices:
                    sampled_labels[image_id] = label

    for image_id, label in sampled_labels.items():
        assert label.startswith("q8rle"), f"Bad label for {image_id}"
        assert q8rle_to_float_matrix(label).shape == expected_shape, f"Shape mismatch for {image_id}"

    print(f"Validated: {len(sorted_ids)} rows, spot-checked {len(sampled_labels)}, shape {expected_shape}")
    print(f"Saved: {zip_path} ({zip_path.stat().st_size / 1024**2:.1f} MB)")